# **Stage 2 — PD Modelling Pipeline**
## US Quarterly | Macro-Driven Delinquency Rate Forecasting
___
**Pipeline overview:**

| Step | Description | Output |
|------|-------------|--------|
| 1 | Data loading from GitHub | — |
| 2 | COVID-19 adjustment: cubic spline + structural break dummy | `delinquency_spline`, `covid_dummy` |
| 3 | OLS regression baseline | `ols_forecast` |
| 4 | Log-OLS alternative | `log_ols_forecast` |
| 5 | Model comparison summary | — |

**Data source:** Reads `SARIMA_regressors_adjusted_US_Q.csv` directly from
`hogandan85/ST-498` GitHub repository — no local path configuration required.

**Target variable:** `us_delinquency_rate` (FRED: DRCCLACBS) — kept in levels per IFRS 9 requirement for PD levels, not changes.

**References:**
Bank & Eder (2021). *A Review on the Probability of Default for IFRS 9.* SSRN 3981339.
Bellini, T. (2019). *IFRS 9 and CECL Credit Risk Modelling and Validation.* Academic Press.
deRitis, C., Licari, J.M. & Ordonez-Sanz, G. (2018). Macroeconomic Forecasting and Scenario Design for IFRS 9 and CECL. In *The New Impairment Model Under IFRS 9 and CECL* (ed. Zhang). Risk Books.
EBA (2021). *IFRS 9 Implementation by EU Institutions — Monitoring Report.* EBA/Rep/2021/35.
Zamil, R. (2020). *Expected loss provisioning under a global pandemic.* FSI Briefs No. 3, BIS.

## **1: Data Loading**
___
`SARIMA_regressors_adjusted_US_Q.csv` is loaded directly from the project GitHub
repository. This file is the single output of `Stage1_SARIMA.ipynb` and
contains 164 rows (144 historical + 20 forecast quarters) and 40 columns —
all raw series, derived variables, lagged regressor columns, and
`us_delinquency_rate` (NaN in forecast rows by definition).

**Prerequisite:** `SARIMA_regressors_adjusted_US_Q.csv` must be committed to
`hogandan85/ST-498 main/Data Collection/` before running this notebook.
Run `Stage1_SARIMA.ipynb` first if the file is not yet in the repository.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ── Data source ───────────────────────────────────────────────────────────────
# Reads SARIMA_regressors_US_Q.csv (Stage 1 output) from GitHub.
# The COVID-adjusted version is created in Section 2 and saved locally.
# No local path configuration needed — runs on any machine.
# Output of Stage1_SARIMA.ipynb — commit that file before running this notebook.
GITHUB_RAW = 'https://raw.githubusercontent.com/hogandan85/ST-498/main/Data%20Collection'

# ── Local output directory ────────────────────────────────────────────────────
# PD forecast outputs are saved here and committed to the repository
LOCAL_OUTPUT = Path(r'C:\Users\andre\OneDrive\LSE\ST498 - Capstone Project\ST-498\Data Collection')

# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv(
    f'{GITHUB_RAW}/SARIMA_regressors_US_Q.csv',
    index_col=0, parse_dates=True
)

print(f'Loaded SARIMA_regressors_US_Q.csv from GitHub')
print(f'  Shape           : {df.shape}')
print(f'  From            : {df.index.min().date()}  to  {df.index.max().date()}')
print(f'  Historical rows : {df["us_delinquency_rate"].notna().sum()}')
print(f'  Forecast rows   : {df["us_delinquency_rate"].isna().sum()}')
print(f'  Columns         : {df.shape[1]}')


Loaded SARIMA_regressors_US_Q.csv from GitHub
  Shape           : (164, 40)
  From            : 1990-03-31  to  2030-12-31
  Historical rows : 140
  Forecast rows   : 24
  Columns         : 40


## **2: COVID-19 Adjustment — Cubic Spline & Structural Break Dummy**
___
Before fitting any PD model, `us_delinquency_rate` is adjusted to correct
for the artificial suppression of defaults during the COVID-19 period
**(2020 Q1 – 2022 Q2)**.

### Why this adjustment is necessary

Government support measures — stimulus payments, mortgage payment holidays,
and moratoria — suppressed observed delinquency rates below their true
economic level during this period. The EBA (2021, §75) explicitly documents
that public support measures *reduced the level of defaults that would
otherwise be observed.* Training a regression model on these artificially
suppressed values biases coefficients downward and produces forecasts that
underestimate default risk in stress scenarios.

### Why COVID only — not other recessions

The 1990s, dot-com, and GFC recessions all followed the standard
macro-default transmission mechanism and represent the most informative
training signal for the model. COVID broke the mechanism through policy
intervention, not through underlying economics. Masking the GFC would
discard the model's most important stress calibration period.

### Two-layer adjustment approach

**Layer 1 — Cubic spline reconstruction**

A natural cubic spline $S(t)$ is fitted through the non-COVID observations
$(t_1, y_1), \ldots, (t_n, y_n)$ and evaluated over the COVID window to
reconstruct a counterfactual delinquency path:

$$\tilde{y}_t = \begin{cases} y_t & t \notin [\text{2020 Q1},\, \text{2022 Q2}] \\ S(t) & t \in [\text{2020 Q1},\, \text{2022 Q2}] \end{cases}$$

This is consistent with the EBA (2021, §90) observation that institutions
applied *smoothing factors to macroeconomic variables underlying IFRS 9
scenarios* to reflect average forecasts during the distortion period.

**Layer 2 — COVID structural break dummy**

A binary dummy variable is included as a regressor in all PD models:

$$D_t^{\text{COVID}} = \begin{cases} 1 & t \in [\text{2020 Q1},\, \text{2022 Q2}] \\ 0 & \text{otherwise} \end{cases}$$

This captures any residual structural break effect not absorbed by the
spline. The temporary nature of the distortion is supported by EBA (2021,
§94), which documents that IFRS 9 PD patterns largely reverted to
pre-COVID relationships by December 2020.

**References:**
EBA (2021). *IFRS 9 Implementation by EU Institutions — Monitoring Report.*
EBA/Rep/2021/35, paragraphs 75, 90, 94.

Zamil, R. (2020). *Expected loss provisioning under a global pandemic.*
FSI Briefs No. 3, Bank for International Settlements.

In [2]:
import pandas as pd
import numpy as np
from scipy.interpolate import CubicSpline
import plotly.graph_objects as go


# ── Load ──────────────────────────────────────────────────────────────────────
GITHUB_RAW   = 'https://raw.githubusercontent.com/hogandan85/ST-498/main/Data%20Collection'
LOCAL_OUTPUT = Path(r'C:\Users\andre\OneDrive\LSE\ST498 - Capstone Project\ST-498\Data Collection')
df = pd.read_csv(f'{GITHUB_RAW}/SARIMA_regressors_US_Q.csv', index_col=0, parse_dates=True)

TARGET = 'us_delinquency_rate'

# ── Define COVID adjustment window ────────────────────────────────────────────
# 2020 Q1 – 2022 Q2: government support suppressed observed defaults
# EBA (2021, §75); Zamil (2020, FSI Briefs No. 3)
COVID_START = '2020-03-31'
COVID_END   = '2022-06-30'
covid_mask  = (df.index >= COVID_START) & (df.index <= COVID_END)

# ── Layer 1: Cubic spline reconstruction ─────────────────────────────────────
# Fit spline on non-COVID, observed quarters only
# EBA (2021, §90): smoothing factors applied to macro variables in IFRS 9 scenarios
train_mask  = df[TARGET].notna() & ~covid_mask
spline_data = df.loc[train_mask, TARGET].copy()

x_all   = np.arange(len(df))
x_train = x_all[train_mask.values]
y_train = spline_data.values

cs = CubicSpline(x_train, y_train)

df['delinquency_spline'] = df[TARGET].copy()
covid_idx = np.where(covid_mask)[0]
df.loc[covid_mask, 'delinquency_spline'] = cs(covid_idx)

# ── Layer 2: COVID structural break dummy ─────────────────────────────────────
# Binary indicator — 1 during COVID distortion window, 0 otherwise
# EBA (2021, §94): distortion temporary, PD patterns reverted by end 2020
df['covid_dummy'] = covid_mask.astype(int)

# ── Save updated dataset ──────────────────────────────────────────────────────
df.to_csv(LOCAL_OUTPUT / 'SARIMA_regressors_adjusted_US_Q.csv')
print(f'Saved SARIMA_regressors_adjusted_US_Q.csv — added delinquency_spline and covid_dummy')
print(f'COVID window : {COVID_START} to {COVID_END}  ({covid_mask.sum()} quarters)')
print(f'Spline range during COVID : {df.loc[covid_mask, "delinquency_spline"].min():.3f}% '
      f'to {df.loc[covid_mask, "delinquency_spline"].max():.3f}%')
print(f'Actual range during COVID : {df.loc[covid_mask, TARGET].min():.3f}% '
      f'to {df.loc[covid_mask, TARGET].max():.3f}%')

# ── Plot: actual vs spline-adjusted ──────────────────────────────────────────
NAVY, AMBER, RED, GREY = '#1F3864', '#E67E22', '#C0392B', '#7F8C8D'
RECESSIONS = [
    ('1990-07-01', '1991-03-31', 'Early 1990s'),
    ('2001-03-01', '2001-12-31', 'Dot-com'),
    ('2007-12-01', '2009-06-30', 'GFC'),
    ('2020-02-01', '2020-04-30', 'COVID'),
]

hist = df[df[TARGET].notna()]

fig = go.Figure()

for s, e, lbl in RECESSIONS:
    fig.add_vrect(x0=s, x1=e, fillcolor='lightgrey', opacity=0.35,
                  layer='below', line_width=0,
                  annotation_text=lbl, annotation_position='top left',
                  annotation_font_size=8, annotation_font_color=GREY)

fig.add_trace(go.Scatter(
    x=hist.index, y=hist[TARGET],
    mode='lines', name='Actual observed',
    line=dict(color=NAVY, width=2)))

fig.add_trace(go.Scatter(
    x=hist.index, y=hist['delinquency_spline'],
    mode='lines', name='Spline-adjusted (COVID window)',
    line=dict(color=AMBER, width=2, dash='dot')))

fig.add_vrect(
    x0=COVID_START, x1=COVID_END,
    fillcolor='rgba(192,57,43,0.08)', line_width=0,
    annotation_text='COVID adjustment window',
    annotation_position='top left',
    annotation_font_size=9, annotation_font_color=RED)

fig.update_layout(
    title=dict(
        text='Figure S2.0 — COVID-19 Adjustment: Actual vs Spline-Reconstructed Delinquency Rate<br>'
             '<sup>Spline fitted on non-COVID observations | '
             'COVID window: 2020 Q1 – 2022 Q2 | '
             'EBA (2021, §75, §90); Zamil (2020, FSI Briefs No. 3)</sup>',
        font=dict(size=13, color=NAVY)),
    xaxis_title='Quarter',
    yaxis_title='Delinquency Rate (%)',
    template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02,
                xanchor='right', x=1, font=dict(size=10)),
    height=500,
    margin=dict(t=100, b=50, l=60, r=30))

fig.show()

Saved SARIMA_regressors_adjusted_US_Q.csv — added delinquency_spline and covid_dummy
COVID window : 2020-03-31 to 2022-06-30  (10 quarters)
Spline range during COVID : 1.845% to 2.578%
Actual range during COVID : 1.530% to 2.690%


## **3: OLS Regression (Baseline)**
___
The first Stage 2 model regresses the spline-adjusted delinquency rate
against the six macroeconomic predictors identified as significant in the
CCF analysis (EDA Section 5), plus the COVID structural break dummy.

### Model specification

$$\tilde{y}_t = \beta_0
+ \beta_1 \cdot \text{HPI}_{t-3}
+ \beta_2 \cdot \text{CC}_{t-2}
+ \beta_3 \cdot \text{UNEMP}_{t-1}
+ \beta_4 \cdot \text{CREDIT}_{t-6}
+ \beta_5 \cdot \text{GDP}_{t}
+ \beta_6 \cdot \text{CPI}_{t-6}
+ \beta_7 \cdot D_t^{\text{COVID}}
+ \varepsilon_t$$

where $\tilde{y}_t$ is the spline-adjusted delinquency rate, subscripts
denote the CCF-optimal lag for each variable (Djeundje & Crook 2019;
Bank & Eder 2021, Section 10), and $D_t^{\text{COVID}}$ is the binary
COVID dummy (1 during 2020 Q1 – 2022 Q2, 0 otherwise).

Estimation is by Ordinary Least Squares:
$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \tilde{\mathbf{y}}$$

OLS is the methodologically appropriate baseline for a continuous
portfolio-level default rate, consistent with the Variable Scalar Approach
framework in Bank & Eder (2021, Section 9.1).

**Training period:** 1991-Q4 to 2025-Q4 (observed target, post-adjustment)
**Forecast period:** 2026-Q1 to 2030-Q4 (regressors from SARIMA Stage 1;
$D_t^{\text{COVID}} = 0$ in forecast period by construction)

In [3]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import plotly.graph_objects as go


# ── Load ──────────────────────────────────────────────────────────────────────
LOCAL_OUTPUT = Path(r'C:\Users\andre\OneDrive\LSE\ST498 - Capstone Project\ST-498\Data Collection')
df = pd.read_csv(LOCAL_OUTPUT / 'SARIMA_regressors_adjusted_US_Q.csv', index_col=0, parse_dates=True)

TARGET = 'delinquency_spline'

REGRESSORS = [
    'us_house_price_yoy_L3',
    'us_consumer_confidence_L2',
    'us_unemployment_L1',
    'us_credit_qoq_growth_L6',
    'us_gdp_yoy_growth_L0',
    'us_cpi_L6',
    'covid_dummy',
]

# ── Split training / forecast ─────────────────────────────────────────────────
# Training: rows where original delinquency rate is observed
# Forecast: rows where it is NaN (2026–2030); covid_dummy = 0 by construction
train    = df[df['us_delinquency_rate'].notna()][REGRESSORS + [TARGET]].dropna()
forecast = df[df['us_delinquency_rate'].isna()][REGRESSORS].dropna()

print(f'Training rows  : {len(train)}  ({train.index[0].date()} to {train.index[-1].date()})')
print(f'Forecast rows  : {len(forecast)}  ({forecast.index[0].date()} to {forecast.index[-1].date()})')

# ── Fit OLS ───────────────────────────────────────────────────────────────────
X_train    = sm.add_constant(train[REGRESSORS])
y_train    = train[TARGET]
X_forecast = sm.add_constant(forecast[REGRESSORS], has_constant='add')

ols_model = sm.OLS(y_train, X_train).fit()
print(ols_model.summary())

# ── In-sample fit ─────────────────────────────────────────────────────────────
train = train.copy()
train['ols_fitted'] = ols_model.fittedvalues

# ── Out-of-sample forecast + CIs ─────────────────────────────────────────────
forecast = forecast.copy()
forecast['ols_forecast'] = ols_model.predict(X_forecast)
pred_df = ols_model.get_prediction(X_forecast).summary_frame(alpha=0.05)
forecast['ols_ci_lower'] = pred_df['obs_ci_lower'].values
forecast['ols_ci_upper'] = pred_df['obs_ci_upper'].values

# ── Plot ──────────────────────────────────────────────────────────────────────
NAVY, BLUE, RED, AMBER, GREY = '#1F3864', '#2E75B6', '#C0392B', '#E67E22', '#7F8C8D'
RECESSIONS = [
    ('1990-07-01', '1991-03-31', 'Early 1990s'),
    ('2001-03-01', '2001-12-31', 'Dot-com'),
    ('2007-12-01', '2009-06-30', 'GFC'),
    ('2020-02-01', '2020-04-30', 'COVID'),
]

fig = go.Figure()

for s, e, lbl in RECESSIONS:
    fig.add_vrect(x0=s, x1=e, fillcolor='lightgrey', opacity=0.35,
                  layer='below', line_width=0,
                  annotation_text=lbl, annotation_position='top left',
                  annotation_font_size=8, annotation_font_color=GREY)

# Actual observed (original, not spline)
hist_obs = df[df['us_delinquency_rate'].notna()]
fig.add_trace(go.Scatter(
    x=hist_obs.index, y=hist_obs['us_delinquency_rate'],
    mode='lines', name='Actual observed',
    line=dict(color=NAVY, width=2)))

# Spline-adjusted target
fig.add_trace(go.Scatter(
    x=train.index, y=train[TARGET],
    mode='lines', name='Spline-adjusted (training target)',
    line=dict(color=AMBER, width=1.5, dash='dot')))

# OLS in-sample fit
fig.add_trace(go.Scatter(
    x=train.index, y=train['ols_fitted'],
    mode='lines', name='OLS fitted (in-sample)',
    line=dict(color=BLUE, width=1.5, dash='dash')))

# Forecast CI band
fig.add_trace(go.Scatter(
    x=list(forecast.index) + list(forecast.index[::-1]),
    y=list(forecast['ols_ci_upper']) + list(forecast['ols_ci_lower'][::-1]),
    fill='toself', fillcolor='rgba(192,57,43,0.10)',
    line=dict(width=0), showlegend=True, name='95% CI (forecast)'))

# Forecast line
fig.add_trace(go.Scatter(
    x=forecast.index, y=forecast['ols_forecast'],
    mode='lines', name='OLS forecast (2026–2030)',
    line=dict(color=RED, width=2, dash='dash')))

fig.add_vline(x=forecast.index[0].timestamp() * 1000,
              line_dash='dash', line_color=GREY, line_width=1,
              annotation_text='Forecast start',
              annotation_font_size=9, annotation_font_color=GREY)

fig.update_layout(
    title=dict(
        text='Figure S2.1 — OLS Baseline: Delinquency Rate Fit & Forecast<br>'
             '<sup>Training: 1991–2025 (spline-adjusted) | Forecast: 2026–2030 | '
             'COVID dummy included | Regressors at CCF-optimal lags</sup>',
        font=dict(size=13, color=NAVY)),
    xaxis_title='Quarter',
    yaxis_title='Delinquency Rate (%)',
    template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02,
                xanchor='right', x=1, font=dict(size=10)),
    height=500,
    margin=dict(t=100, b=50, l=60, r=30))

fig.show()

print('\nOLS Forecast — 2026 Q1 to 2030 Q4:')
print(forecast[['ols_forecast', 'ols_ci_lower', 'ols_ci_upper']].round(4).to_string())

Training rows  : 137  (1991-12-31 to 2025-12-31)
Forecast rows  : 19  (2026-03-31 to 2030-12-31)
                            OLS Regression Results                            
Dep. Variable:     delinquency_spline   R-squared:                       0.628
Model:                            OLS   Adj. R-squared:                  0.608
Method:                 Least Squares   F-statistic:                     31.11
Date:                Tue, 14 Apr 2026   Prob (F-statistic):           6.82e-25
Time:                        19:22:55   Log-Likelihood:                -145.24
No. Observations:                 137   AIC:                             306.5
Df Residuals:                     129   BIC:                             329.8
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                                coef    std err          t      P>|t|      [0.025      0.975]
-------------------


OLS Forecast — 2026 Q1 to 2030 Q4:
            ols_forecast  ols_ci_lower  ols_ci_upper
2026-03-31        3.3266        1.8834        4.7699
2026-06-30        2.4739        1.0000        3.9479
2026-09-30        3.0492        1.5906        4.5077
2026-12-31        3.3092        1.8566        4.7618
2027-03-31        3.5229        2.0783        4.9676
2027-09-30        4.0749        2.6355        5.5142
2027-12-31        3.5963        2.1533        5.0394
2028-03-31        3.5069        2.0629        4.9509
2028-06-30        3.4025        1.9566        4.8485
2028-09-30        3.5239        2.0800        4.9677
2028-12-31        3.6245        2.1822        5.0668
2029-03-31        3.5244        2.0808        4.9681
2029-06-30        3.4313        1.9861        4.8764
2029-09-30        3.5421        2.0988        4.9854
2029-12-31        3.6369        2.1949        5.0790
2030-03-31        3.5462        2.1029        4.9894
2030-06-30        3.4613        2.0167        4.9060
2030-09-30

## **4: Log-Transformed OLS (Alternative)**
___
The log-OLS model addresses two limitations of the plain OLS baseline:
the right-skewed distribution of the delinquency rate and the possibility
of predicting negative values. Taking the natural log of the target
linearises the multiplicative relationship between macro drivers and the
default rate, and constrains back-transformed forecasts to positive values
by construction.

A fractional logit (GLM, logit link, binomial family) was evaluated as an
alternative but rejected: applying a binomial GLM to a rate bounded in the
narrow range 1.53%–6.77% produced near-zero pseudo R², insignificant
coefficients, and confidence intervals reaching 33% — well outside the
historical maximum. This is a known failure mode of the binomial GLM when
proportions are far from 0.5 (Bank & Eder 2021, Section 6.2).

### Model specification

$$\log(\tilde{y}_t) = \beta_0
+ \beta_1 \cdot \text{HPI}_{t-3}
+ \beta_2 \cdot \text{CC}_{t-2}
+ \beta_3 \cdot \text{UNEMP}_{t-1}
+ \beta_4 \cdot \text{CREDIT}_{t-6}
+ \beta_5 \cdot \text{GDP}_{t}
+ \beta_6 \cdot \text{CPI}_{t-6}
+ \beta_7 \cdot D_t^{\text{COVID}}
+ \varepsilon_t$$

Forecasts are back-transformed to the original scale via:
$$\hat{y}_t = \exp\!\left(\hat{\beta}_0 + \mathbf{x}_t^\top \hat{\boldsymbol{\beta}}\right)$$

This is consistent with the treatment of bounded economic rates in
Bank & Eder (2021, Section 9.1) and Bellini (2019, p. 257).

**Same regressors, lag structure, and COVID adjustment as OLS —
direct comparison of the two models is therefore valid.**

**Training period:** 1991-Q4 to 2025-Q4
**Forecast period:** 2026-Q1 to 2030-Q4

In [4]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import plotly.graph_objects as go


# ── Load ──────────────────────────────────────────────────────────────────────
LOCAL_OUTPUT = Path(r'C:\Users\andre\OneDrive\LSE\ST498 - Capstone Project\ST-498\Data Collection')
df = pd.read_csv(LOCAL_OUTPUT / 'SARIMA_regressors_adjusted_US_Q.csv', index_col=0, parse_dates=True)

TARGET = 'delinquency_spline'

REGRESSORS = [
    'us_house_price_yoy_L3',
    'us_consumer_confidence_L2',
    'us_unemployment_L1',
    'us_credit_qoq_growth_L6',
    'us_gdp_yoy_growth_L0',
    'us_cpi_L6',
    'covid_dummy',
]

# ── Split ─────────────────────────────────────────────────────────────────────
train    = df[df['us_delinquency_rate'].notna()][REGRESSORS + [TARGET]].dropna().copy()
forecast = df[df['us_delinquency_rate'].isna()][REGRESSORS].dropna().copy()

print(f'Training rows  : {len(train)}  ({train.index[0].date()} to {train.index[-1].date()})')
print(f'Forecast rows  : {len(forecast)}  ({forecast.index[0].date()} to {forecast.index[-1].date()})')

# ── Log-transform target ──────────────────────────────────────────────────────
train['log_delinquency'] = np.log(train[TARGET])

# ── Fit Log-OLS ───────────────────────────────────────────────────────────────
X_train    = sm.add_constant(train[REGRESSORS])
X_forecast = sm.add_constant(forecast[REGRESSORS], has_constant='add')

log_ols = sm.OLS(train['log_delinquency'], X_train).fit()
print(log_ols.summary())

# ── In-sample fit (back-transformed) ─────────────────────────────────────────
train['log_ols_fitted'] = np.exp(log_ols.fittedvalues)

# ── Out-of-sample forecast (back-transformed) ─────────────────────────────────
forecast['log_ols_forecast'] = np.exp(log_ols.predict(X_forecast))

# ── CIs (back-transformed) ────────────────────────────────────────────────────
pred_df = log_ols.get_prediction(X_forecast).summary_frame(alpha=0.05)
forecast['log_ols_ci_lower'] = np.exp(pred_df['obs_ci_lower'].values)
forecast['log_ols_ci_upper'] = np.exp(pred_df['obs_ci_upper'].values)

# ── In-sample diagnostics (original % scale) ──────────────────────────────────
actual_orig = train[TARGET]
fitted_orig = train['log_ols_fitted']
rmse_orig   = np.sqrt(np.mean((actual_orig - fitted_orig) ** 2))
mae_orig    = np.mean(np.abs(actual_orig - fitted_orig))

print(f'\nIn-sample diagnostics (original % scale):')
print(f'  RMSE  : {rmse_orig:.4f}%')
print(f'  MAE   : {mae_orig:.4f}%')
print(f'  R²    : {log_ols.rsquared:.4f}')
print(f'  Adj R²: {log_ols.rsquared_adj:.4f}')

# ── Plot ──────────────────────────────────────────────────────────────────────
NAVY, BLUE, AMBER, GREY = '#1F3864', '#2E75B6', '#E67E22', '#7F8C8D'
RECESSIONS = [
    ('1990-07-01', '1991-03-31', 'Early 1990s'),
    ('2001-03-01', '2001-12-31', 'Dot-com'),
    ('2007-12-01', '2009-06-30', 'GFC'),
    ('2020-02-01', '2020-04-30', 'COVID'),
]

fig = go.Figure()

for s, e, lbl in RECESSIONS:
    fig.add_vrect(x0=s, x1=e, fillcolor='lightgrey', opacity=0.35,
                  layer='below', line_width=0,
                  annotation_text=lbl, annotation_position='top left',
                  annotation_font_size=8, annotation_font_color=GREY)

# Actual observed (original)
hist_obs = df[df['us_delinquency_rate'].notna()]
fig.add_trace(go.Scatter(
    x=hist_obs.index, y=hist_obs['us_delinquency_rate'],
    mode='lines', name='Actual observed',
    line=dict(color=NAVY, width=2)))

# Log-OLS in-sample fit
fig.add_trace(go.Scatter(
    x=train.index, y=train['log_ols_fitted'],
    mode='lines', name='Log-OLS fitted (in-sample)',
    line=dict(color=BLUE, width=1.5, dash='dot')))

# Forecast CI band
fig.add_trace(go.Scatter(
    x=list(forecast.index) + list(forecast.index[::-1]),
    y=list(forecast['log_ols_ci_upper']) + list(forecast['log_ols_ci_lower'][::-1]),
    fill='toself', fillcolor='rgba(230,126,34,0.10)',
    line=dict(width=0), showlegend=True, name='95% CI (forecast)'))

# Forecast line
fig.add_trace(go.Scatter(
    x=forecast.index, y=forecast['log_ols_forecast'],
    mode='lines', name='Log-OLS forecast (2026–2030)',
    line=dict(color=AMBER, width=2, dash='dash')))

fig.add_vline(x=forecast.index[0].timestamp() * 1000,
              line_dash='dash', line_color=GREY, line_width=1,
              annotation_text='Forecast start',
              annotation_font_size=9, annotation_font_color=GREY)

fig.update_layout(
    title=dict(
        text='Figure S2.2 — Log-OLS: Delinquency Rate Fit & Forecast<br>'
             '<sup>Training: 1991–2025 (spline-adjusted) | Forecast: 2026–2030 | '
             'Fitted on log(rate), back-transformed with exp() for display</sup>',
        font=dict(size=13, color=NAVY)),
    xaxis_title='Quarter',
    yaxis_title='Delinquency Rate (%)',
    template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02,
                xanchor='right', x=1, font=dict(size=10)),
    height=500,
    margin=dict(t=100, b=50, l=60, r=30))

fig.show()

print('\nLog-OLS Forecast — 2026 Q1 to 2030 Q4:')
print(forecast[['log_ols_forecast', 'log_ols_ci_lower',
                'log_ols_ci_upper']].round(4).to_string())

Training rows  : 137  (1991-12-31 to 2025-12-31)
Forecast rows  : 19  (2026-03-31 to 2030-12-31)
                            OLS Regression Results                            
Dep. Variable:        log_delinquency   R-squared:                       0.610
Model:                            OLS   Adj. R-squared:                  0.589
Method:                 Least Squares   F-statistic:                     28.87
Date:                Tue, 14 Apr 2026   Prob (F-statistic):           1.26e-23
Time:                        19:23:07   Log-Likelihood:                 24.321
No. Observations:                 137   AIC:                            -32.64
Df Residuals:                     129   BIC:                            -9.282
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                                coef    std err          t      P>|t|      [0.025      0.975]
-------------------


Log-OLS Forecast — 2026 Q1 to 2030 Q4:
            log_ols_forecast  log_ols_ci_lower  log_ols_ci_upper
2026-03-31            3.2030            2.1074            4.8682
2026-06-30            2.5539            1.6654            3.9163
2026-09-30            2.9790            1.9514            4.5479
2026-12-31            3.1932            2.0953            4.8666
2027-03-31            3.3814            2.2238            5.1414
2027-09-30            3.9095            2.5751            5.9352
2027-12-31            3.4332            2.2590            5.2178
2028-03-31            3.3551            2.2069            5.1005
2028-06-30            3.2605            2.1435            4.9595
2028-09-30            3.3676            2.2153            5.1193
2028-12-31            3.4604            2.2774            5.2579
2029-03-31            3.3687            2.2162            5.1206
2029-06-30            3.2854            2.1604            4.9961
2029-09-30            3.3846            2.2268    

## **5: Model Comparison Summary**
___

In [6]:
import pandas as pd
import numpy as np
import statsmodels.api as sm


# ── Reload dataset and recompute forecast inputs ──────────────────────────────
LOCAL_OUTPUT = Path(r'C:\Users\andre\OneDrive\LSE\ST498 - Capstone Project\ST-498\Data Collection')
df = pd.read_csv(LOCAL_OUTPUT / 'SARIMA_regressors_adjusted_US_Q.csv', index_col=0, parse_dates=True)

TARGET = 'delinquency_spline'
REGRESSORS = [
    'us_house_price_yoy_L3',
    'us_consumer_confidence_L2',
    'us_unemployment_L1',
    'us_credit_qoq_growth_L6',
    'us_gdp_yoy_growth_L0',
    'us_cpi_L6',
    'covid_dummy',
]

forecast_df = df[df['us_delinquency_rate'].isna()][REGRESSORS].dropna().copy()
X_forecast  = sm.add_constant(forecast_df[REGRESSORS], has_constant='add')

# ── Recompute forecast values from model objects ──────────────────────────────
ols_fc      = ols_model.predict(X_forecast)
ols_ci      = ols_model.get_prediction(X_forecast).summary_frame(alpha=0.05)

log_fc      = np.exp(log_ols.predict(X_forecast))
log_ci_raw  = log_ols.get_prediction(X_forecast).summary_frame(alpha=0.05)
log_ci_lo   = np.exp(log_ci_raw['obs_ci_lower'].values)
log_ci_hi   = np.exp(log_ci_raw['obs_ci_upper'].values)

# ── In-sample diagnostics ─────────────────────────────────────────────────────
def model_diagnostics(model, fitted_orig, actual_orig, model_name):
    resid_orig = actual_orig - fitted_orig
    return {
        'Model':          model_name,
        'R²':             round(model.rsquared, 3),
        'Adj R²':         round(model.rsquared_adj, 3),
        'Durbin-Watson':  round(float(sm.stats.stattools.durbin_watson(model.resid)), 3),
        'RMSE (%)':       round(float(np.sqrt(np.mean(resid_orig ** 2))), 4),
        'MAE (%)':        round(float(np.mean(np.abs(resid_orig))), 4),
        'AIC':            round(model.aic, 1),
        'BIC':            round(model.bic, 1),
        'N (training)':   int(model.nobs),
    }

# Actual values from model endog
ols_actual = ols_model.model.endog
log_actual = np.exp(log_ols.model.endog)

rows = [
    model_diagnostics(ols_model, ols_model.fittedvalues,     ols_actual, 'OLS (spline + dummy)'),
    model_diagnostics(log_ols,   np.exp(log_ols.fittedvalues), log_actual, 'Log-OLS (spline + dummy)'),
]

comparison_table = pd.DataFrame(rows).set_index('Model')
print('Model Comparison — In-Sample Diagnostics')
print(comparison_table.to_string())

# ── Forecast range summary ────────────────────────────────────────────────────
print('\nForecast Range Summary (2026–2030):')
forecast_summary = pd.DataFrame({
    'Model':            ['OLS', 'Log-OLS'],
    'Forecast min (%)': [round(ols_fc.min(), 3),              round(log_fc.min(), 3)],
    'Forecast mean (%)': [round(ols_fc.mean(), 3),            round(log_fc.mean(), 3)],
    'Forecast max (%)': [round(ols_fc.max(), 3),              round(log_fc.max(), 3)],
    'CI lower mean (%)': [round(ols_ci['obs_ci_lower'].mean(), 3), round(log_ci_lo.mean(), 3)],
    'CI upper mean (%)': [round(ols_ci['obs_ci_upper'].mean(), 3), round(log_ci_hi.mean(), 3)],
}).set_index('Model')
print(forecast_summary.to_string())

# ── COVID dummy significance ──────────────────────────────────────────────────
print('\nCOVID Dummy Coefficient:')
for name, model in [('OLS', ols_model), ('Log-OLS', log_ols)]:
    coef = model.params['covid_dummy']
    pval = model.pvalues['covid_dummy']
    sig  = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'n.s.'
    print(f'  {name:<25s}  coef={coef:+.4f}  p={pval:.4f}  {sig}')
# ── Save PD forecast results ─────────────────────────────────────────────────
pd_forecasts = pd.DataFrame({
    'ols_forecast':     ols_fc.values,
    'ols_ci_lower':     ols_ci['obs_ci_lower'].values,
    'ols_ci_upper':     ols_ci['obs_ci_upper'].values,
    'log_ols_forecast': log_fc.values,
    'log_ols_ci_lower': log_ci_lo,
    'log_ols_ci_upper': log_ci_hi,
}, index=forecast_df.index)

out_path = LOCAL_OUTPUT / 'SARIMA_PD_forecasts_US_Q.csv'
pd_forecasts.to_csv(out_path)
print(f'\nSaved SARIMA_PD_forecasts_US_Q.csv  {pd_forecasts.shape}')
print(f'  Path : {out_path}')
print(pd_forecasts.round(4).to_string())


Model Comparison — In-Sample Diagnostics
                             R²  Adj R²  Durbin-Watson  RMSE (%)  MAE (%)    AIC    BIC  N (training)
Model                                                                                                
OLS (spline + dummy)      0.628   0.608          0.671    0.6985   0.5771  306.5  329.8           137
Log-OLS (spline + dummy)  0.610   0.589          0.610    0.6873   0.5769  -32.6   -9.3           137

Forecast Range Summary (2026–2030):
         Forecast min (%)  Forecast mean (%)  Forecast max (%)  CI lower mean (%)  CI upper mean (%)
Model                                                                                               
OLS                 2.474              3.461             4.075              2.015              4.908
Log-OLS             2.554              3.326             3.909              2.187              5.058

COVID Dummy Coefficient:
  OLS                        coef=-1.0489  p=0.0002  ***
  Log-OLS                  

### References

Bank, M. & Eder, B. (2021). *A Review on the Probability of Default for
IFRS 9.* SSRN 3981339.

Bellini, T. (2019). *IFRS 9 and CECL Credit Risk Modelling and Validation.*
Academic Press, pp. 257, 276.

deRitis, C., Licari, J.M. & Ordonez-Sanz, G. (2018). Macroeconomic
Forecasting and Scenario Design for IFRS 9 and CECL. In *The New Impairment
Model Under IFRS 9 and CECL* (ed. Zhang). Risk Books.

Djeundje, V.B. & Crook, J. (2019). Dynamic survival models with varying
coefficients for credit risks. *European Journal of Operational Research*,
275(1), 319–333.

EBA (2021). *IFRS 9 Implementation by EU Institutions — Monitoring Report.*
EBA/Rep/2021/35, paragraphs 75, 90, 94.

Zamil, R. (2020). *Expected loss provisioning under a global pandemic.*
FSI Briefs No. 3, Bank for International Settlements.